RAG-Based Solar Inverter Document Question Answering System

To develop and execute a beginner-friendly Retrieval-Augmented Generation (RAG) model to fetch relevant information from a Solar Inverter Project PDF document using sentence embeddings, FAISS vector search, and a local question-answering model to generate precise document-based answers

In [5]:
!pip install sentence-transformers faiss-cpu transformers pypdf

In [19]:
from pypdf import PdfReader

pdf_path = "solar_inverter_project_document.pdf"
reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += page.extract_text()

print(text[:1000])

IJAR
SCT
 
 
ISSN (Online) 2581
-
9429
 
 
 
 
 
 
       
International Journal of Advanced 
Research in Science
, Communication and
 
Technology
 
(IJARSCT)
 
                             
International Open
-
Access, Double
-
Blind, Peer
-
Reviewed, 
Refereed, Multidisciplinary Online Journal
 
 
Volume 
3
, Issue
 
9
, 
May 2023
 
Copyright to IJAR
S
C
T
 
 
DOI: 10.48175/
IJARSCT
-
103
9
3
 
              
 
498
 
www.ijarsct.co.in
                                                
 
 
 
Impact Factor: 
7.301
 
Solar
 
Inverter
 
Project
 
Monu Ranjan, Suraj Shivaji Bidgar, Patil Pratik Satish
, 
 
Waikar Aniket Bibhishan
, 
Prof. Manohar Kalgonde
 
Sinhgad 
Institute of
 
Technology
, 
Lonavala
,
 
Maharashtra, India
 
 
Abstract
:
 
This project aims to design and implement a solar inverter system that generates 
pollution
-
free 
electricity from solar energy during the day and stores it in a battery for use during the night or in 
transportation vehicles. The system consists of 

In [2]:
# Simple text cleaning
text = text.replace("\n", " ")
chunk_size = 500# Chunking
chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])
    
print("Total Chunks:", len(chunks))
print(chunks[0])

Total Chunks: 22
IJAR SCT     ISSN (Online) 2581 - 9429                     International Journal of Advanced  Research in Science , Communication and   Technology   (IJARSCT)                                 International Open - Access, Double - Blind, Peer - Reviewed,  Refereed, Multidisciplinary Online Journal     Volume  3 , Issue   9 ,  May 2023   Copyright to IJAR S C T     DOI: 10.48175/ IJARSCT - 103 9 3                    498   www.ijarsct.co.in                                                        Impa


In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")#Load embedding model
chunk_embeddings = embed_model.encode(chunks)#Create embeddings
chunk_embeddings = np.array(chunk_embeddings)#Convert to numpy array
print("Embedding shape:", chunk_embeddings.shape)

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 211.96it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (22, 384)


In [5]:
dimension = chunk_embeddings.shape[1]# Get embedding dimension
index = faiss.IndexFlatL2(dimension)# Create FAISS index
index.add(chunk_embeddings)# Add embeddings to index

print("Total vectors in index:", index.ntotal)

Total vectors in index: 22


In [6]:
def retrieve_chunks(query, top_k=3):#Convert query to embedding
    query_embedding = embed_model.encode([query])
    query_embedding = np.array(query_embedding)
    distances, indices = index.search(query_embedding, top_k)#Search FAISS
    retrieved_chunks = [chunks[i] for i in indices[0]]#Get relevant chunks
    return retrieved_chunks

In [7]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering",
    model="distilbert-base-uncased-distilled-squad")

Loading weights: 100%|███████████████████████| 102/102 [00:00<00:00, 203.18it/s, Materializing param=qa_outputs.weight]


In [8]:
def ask_question(query):
    retrieved_chunks = retrieve_chunks(query)
    context = " ".join(retrieved_chunks)
    result = qa_pipeline(question=query,context=context)
    print("Answer:", result["answer"])

In [9]:
ask_question("What is the purpose of the solar inverter project?")

Answer: generating and using solar power in rural  and tribal areas


In [10]:
ask_question("What components are used in this project?")

Answer: Reliable Power Supply:   ower their homes, vehicles, and devices


In this project, we have been able to develop a beginner-level RAG system with local embedding models and FAISS for document retrieval. The system is able to extract relevant text chunks from a Solar Inverter Project PDF and answer questions using a question-answering model. 